In [ ]:
"""
Economic Dashboard AI Assistant
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Run this INSIDE Snowflake → Streamlit (SiS)
NO external connection needed — runs natively in Snowflake
Table: ECON_AGENT_DB.ANALYTICS.ECONOMIC_DASHBOARD_LIVE
Columns: DATE, CPI, UNEMPLOYMENT_RATE, MORTGAGE_RATE_30Y
"""

import streamlit as st
import pandas as pd
import re
from snowflake.snowpark.context import get_active_session

# ─────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────
st.set_page_config(
    page_title="Economic AI Assistant",
    page_icon="📊",
    layout="wide",
)

# ─────────────────────────────────────────
# CUSTOM CSS
# ─────────────────────────────────────────
st.markdown("""
<style>
    .stApp { background-color: #0f1117; color: #e8eaf6; }
    [data-testid="stSidebar"] { background-color: #161b27; border-right: 1px solid #2a2f45; }

    .user-msg {
        background: linear-gradient(135deg, #1e3a5f, #1565c0);
        border-radius: 18px 18px 4px 18px;
        padding: 14px 18px; margin: 8px 0 8px 60px;
        color: #e8eaf6; font-size: 15px;
        box-shadow: 0 2px 8px rgba(21,101,192,0.3);
    }
    .ai-msg {
        background: linear-gradient(135deg, #1a1f2e, #232840);
        border: 1px solid #2e3558;
        border-radius: 18px 18px 18px 4px;
        padding: 14px 18px; margin: 8px 60px 8px 0;
        color: #e8eaf6; font-size: 15px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.3);
    }
    .ai-msg strong { color: #82b1ff; }
    .metric-chip {
        display: inline-block; background: #1e3a5f;
        border: 1px solid #1565c0; border-radius: 8px;
        padding: 2px 10px; margin: 3px;
        font-weight: 600; color: #90caf9;
    }
    .kpi-card {
        background: linear-gradient(135deg, #1a1f2e, #1e2540);
        border: 1px solid #2e3558; border-radius: 12px;
        padding: 16px 20px; text-align: center; margin-bottom: 8px;
    }
    .kpi-label { font-size: 12px; color: #7986cb; text-transform: uppercase; letter-spacing: 1px; }
    .kpi-value { font-size: 26px; font-weight: 700; color: #82b1ff; }
    .stTextInput > div > div > input {
        background-color: #1a1f2e !important;
        border: 1px solid #2e3558 !important;
        color: #e8eaf6 !important;
        border-radius: 12px !important;
    }
    .stButton > button {
        background: linear-gradient(135deg, #1565c0, #0d47a1) !important;
        color: white !important; border: none !important;
        border-radius: 10px !important; padding: 8px 20px !important;
        font-weight: 600 !important;
    }
    .header-bar {
        background: linear-gradient(90deg, #0d1b2a, #1a237e);
        border-bottom: 1px solid #283593; border-radius: 12px;
        padding: 16px 24px; margin-bottom: 20px;
        display: flex; align-items: center; gap: 12px;
    }
    .info-box {
        background: #1a1f2e; border: 1px solid #2e3558;
        border-radius: 10px; padding: 12px 16px; margin: 6px 0;
        font-size: 13px; color: #7986cb;
    }
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────
# SNOWFLAKE SESSION  (native — no login needed)
# ─────────────────────────────────────────
TABLE_PATH = "ECON_AGENT_DB.ANALYTICS.ECONOMIC_DASHBOARD_LIVE"

@st.cache_resource
def get_session():
    """Get native Snowflake session — works automatically inside Snowflake Streamlit."""
    return get_active_session()

def run_query(sql: str) -> pd.DataFrame:
    """Run SQL using native Snowpark session and return pandas DataFrame."""
    session = get_session()
    return session.sql(sql).to_pandas()


# ─────────────────────────────────────────
# RULE-BASED AI ENGINE
# ─────────────────────────────────────────
def classify_intent(question: str):
    q = question.lower().strip()

    # Latest / current
    if re.search(r"\b(latest|current|recent|today|now|last)\b", q):
        if re.search(r"\bcpi\b", q):
            return ("latest_single", {"col": "CPI", "label": "CPI"})
        if re.search(r"\bunemploy", q):
            return ("latest_single", {"col": "UNEMPLOYMENT_RATE", "label": "Unemployment Rate"})
        if re.search(r"\bmortgage", q):
            return ("latest_single", {"col": "MORTGAGE_RATE_30Y", "label": "30-Year Mortgage Rate"})
        return ("latest_all", {})

    # Trend / history
    if re.search(r"\btrend|over time|change|month|history|historical\b", q):
        months = re.search(r"(\d+)\s*month", q)
        n = int(months.group(1)) if months else 12
        if re.search(r"\bcpi\b", q):
            return ("trend", {"col": "CPI", "label": "CPI", "months": n})
        if re.search(r"\bunemploy", q):
            return ("trend", {"col": "UNEMPLOYMENT_RATE", "label": "Unemployment Rate", "months": n})
        if re.search(r"\bmortgage", q):
            return ("trend", {"col": "MORTGAGE_RATE_30Y", "label": "30-Year Mortgage Rate", "months": n})
        return ("trend_all", {"months": n})

    # Highest / peak
    if re.search(r"\bhighest|peak|max|maximum|top\b", q):
        if re.search(r"\bcpi\b", q):
            return ("peak", {"col": "CPI", "label": "CPI"})
        if re.search(r"\bunemploy", q):
            return ("peak", {"col": "UNEMPLOYMENT_RATE", "label": "Unemployment Rate"})
        if re.search(r"\bmortgage", q):
            return ("peak", {"col": "MORTGAGE_RATE_30Y", "label": "30-Year Mortgage Rate"})
        return ("peak_all", {})

    # Lowest / min
    if re.search(r"\blowest|min|minimum|bottom\b", q):
        if re.search(r"\bcpi\b", q):
            return ("low", {"col": "CPI", "label": "CPI"})
        if re.search(r"\bunemploy", q):
            return ("low", {"col": "UNEMPLOYMENT_RATE", "label": "Unemployment Rate"})
        if re.search(r"\bmortgage", q):
            return ("low", {"col": "MORTGAGE_RATE_30Y", "label": "30-Year Mortgage Rate"})
        return ("low_all", {})

    # Average
    if re.search(r"\baverage|avg|mean\b", q):
        months = re.search(r"(\d+)\s*month", q)
        n = int(months.group(1)) if months else None
        if re.search(r"\bcpi\b", q):
            return ("average", {"col": "CPI", "label": "CPI", "months": n})
        if re.search(r"\bunemploy", q):
            return ("average", {"col": "UNEMPLOYMENT_RATE", "label": "Unemployment Rate", "months": n})
        if re.search(r"\bmortgage", q):
            return ("average", {"col": "MORTGAGE_RATE_30Y", "label": "30-Year Mortgage Rate", "months": n})
        return ("average_all", {"months": n})

    # Year specific
    year_match = re.search(r"\b(20\d{2})\b", q)
    if year_match:
        year = year_match.group(1)
        if re.search(r"\bcpi\b", q):
            return ("year_data", {"col": "CPI", "label": "CPI", "year": year})
        if re.search(r"\bunemploy", q):
            return ("year_data", {"col": "UNEMPLOYMENT_RATE", "label": "Unemployment Rate", "year": year})
        if re.search(r"\bmortgage", q):
            return ("year_data", {"col": "MORTGAGE_RATE_30Y", "label": "30-Year Mortgage Rate", "year": year})
        return ("year_all", {"year": year})

    # Compare
    if re.search(r"\bcompare|vs|versus|correlation|relation\b", q):
        return ("compare_all", {})

    # Summary
    if re.search(r"\bsummary|overview|snapshot|dashboard|tell me|show me|all\b", q):
        return ("summary", {})

    # Meta / count
    if re.search(r"\bhow many|count|records|rows|range|date range\b", q):
        return ("meta", {})

    return ("summary", {})


def build_sql(intent: str, params: dict):
    T     = TABLE_PATH
    col   = params.get("col", "CPI")
    label = params.get("label", col)
    months= params.get("months", 12)
    year  = params.get("year")

    if intent == "latest_single":
        return (f"SELECT DATE, {col} FROM {T} ORDER BY DATE DESC LIMIT 1",
                f"Most recent {label}")

    if intent == "latest_all":
        return (f"SELECT DATE, CPI, UNEMPLOYMENT_RATE, MORTGAGE_RATE_30Y FROM {T} ORDER BY DATE DESC LIMIT 1",
                "Latest all indicators")

    if intent == "trend":
        return (f"SELECT DATE, {col} FROM {T} ORDER BY DATE DESC LIMIT {months}",
                f"{months}-month trend for {label}")

    if intent == "trend_all":
        return (f"SELECT DATE, CPI, UNEMPLOYMENT_RATE, MORTGAGE_RATE_30Y FROM {T} ORDER BY DATE DESC LIMIT {months}",
                f"{months}-month trend all indicators")

    if intent == "peak":
        return (f"SELECT DATE, {col} FROM {T} ORDER BY {col} DESC LIMIT 1",
                f"Peak {label}")

    if intent == "peak_all":
        return (f"""SELECT
                    MAX(CPI)              AS MAX_CPI,
                    MAX(UNEMPLOYMENT_RATE) AS MAX_UNEMPLOYMENT,
                    MAX(MORTGAGE_RATE_30Y) AS MAX_MORTGAGE
                FROM {T}""",
                "All-time peaks")

    if intent == "low":
        return (f"SELECT DATE, {col} FROM {T} ORDER BY {col} ASC LIMIT 1",
                f"Lowest {label}")

    if intent == "low_all":
        return (f"""SELECT
                    MIN(CPI)              AS MIN_CPI,
                    MIN(UNEMPLOYMENT_RATE) AS MIN_UNEMPLOYMENT,
                    MIN(MORTGAGE_RATE_30Y) AS MIN_MORTGAGE
                FROM {T}""",
                "All-time lows")

    if intent == "average":
        where = f"WHERE YEAR(DATE) = {year}" if year else (
                f"WHERE DATE >= DATEADD(MONTH, -{months}, CURRENT_DATE())" if months else "")
        period = f"in {year}" if year else (f"last {months} months" if months else "overall")
        return (f"SELECT ROUND(AVG({col}), 4) AS AVG_{col} FROM {T} {where}",
                f"Average {label} {period}")

    if intent == "average_all":
        where = f"WHERE DATE >= DATEADD(MONTH, -{months}, CURRENT_DATE())" if months else ""
        period = f"last {months} months" if months else "overall"
        return (f"""SELECT
                    ROUND(AVG(CPI), 4)               AS AVG_CPI,
                    ROUND(AVG(UNEMPLOYMENT_RATE), 4)  AS AVG_UNEMPLOYMENT,
                    ROUND(AVG(MORTGAGE_RATE_30Y), 4)  AS AVG_MORTGAGE
                FROM {T} {where}""",
                f"Averages {period}")

    if intent == "year_data":
        return (f"SELECT DATE, {col} FROM {T} WHERE YEAR(DATE) = {year} ORDER BY DATE",
                f"{label} for {year}")

    if intent == "year_all":
        return (f"SELECT DATE, CPI, UNEMPLOYMENT_RATE, MORTGAGE_RATE_30Y FROM {T} WHERE YEAR(DATE) = {year} ORDER BY DATE",
                f"All indicators for {year}")

    if intent == "compare_all":
        return (f"SELECT DATE, CPI, UNEMPLOYMENT_RATE, MORTGAGE_RATE_30Y FROM {T} ORDER BY DATE DESC LIMIT 24",
                "Compare all indicators")

    if intent == "meta":
        return (f"""SELECT
                    COUNT(*)     AS TOTAL_ROWS,
                    MIN(DATE)    AS EARLIEST_DATE,
                    MAX(DATE)    AS LATEST_DATE
                FROM {T}""",
                "Dataset info")

    # default summary
    return (f"SELECT DATE, CPI, UNEMPLOYMENT_RATE, MORTGAGE_RATE_30Y FROM {T} ORDER BY DATE DESC LIMIT 12",
            "12-month snapshot")


def format_response(intent, params, df, explanation):
    if df.empty:
        return "⚠️ No data found for that query."

    col   = params.get("col", "CPI")
    label = params.get("label", col)

    def fmt(val, d=2):
        try:    return f"{float(val):,.{d}f}"
        except: return str(val)

    if intent == "latest_single":
        row = df.iloc[0]
        return (f"📅 **As of {row['DATE']}**\n\n"
                f"Latest **{label}**: <span class='metric-chip'>{fmt(row[col])}</span>")

    if intent == "latest_all":
        row = df.iloc[0]
        return (f"📅 **Latest Snapshot — {row['DATE']}**\n\n"
                f"- 📈 CPI: <span class='metric-chip'>{fmt(row['CPI'])}</span>\n"
                f"- 👷 Unemployment: <span class='metric-chip'>{fmt(row['UNEMPLOYMENT_RATE'])}%</span>\n"
                f"- 🏠 Mortgage 30Y: <span class='metric-chip'>{fmt(row['MORTGAGE_RATE_30Y'])}%</span>")

    if intent == "trend":
        first, last = df.iloc[-1][col], df.iloc[0][col]
        delta = float(last) - float(first)
        arrow = "📈" if delta > 0 else "📉"
        return (f"{arrow} **{label} Trend**\n\n"
                f"- Start : **{fmt(first)}** ({df.iloc[-1]['DATE']})\n"
                f"- Latest: **{fmt(last)}** ({df.iloc[0]['DATE']})\n"
                f"- Change: **{'+' if delta>=0 else ''}{fmt(delta)}**\n\n"
                "Full data in table below ↓")

    if intent == "trend_all":
        old, new = df.iloc[-1], df.iloc[0]
        return (f"📊 **Trends — {len(df)} months**\n\n"
                f"| Indicator | {old['DATE']} | {new['DATE']} |\n|---|---|---|\n"
                f"| CPI | {fmt(old['CPI'])} | {fmt(new['CPI'])} |\n"
                f"| Unemployment | {fmt(old['UNEMPLOYMENT_RATE'])}% | {fmt(new['UNEMPLOYMENT_RATE'])}% |\n"
                f"| Mortgage 30Y | {fmt(old['MORTGAGE_RATE_30Y'])}% | {fmt(new['MORTGAGE_RATE_30Y'])}% |")

    if intent == "peak":
        row = df.iloc[0]
        return f"🔺 **Highest {label}**: <span class='metric-chip'>{fmt(row[col])}</span> on **{row['DATE']}**"

    if intent == "peak_all":
        row = df.iloc[0]
        return (f"🔺 **All-Time Peaks**\n\n"
                f"- CPI: <span class='metric-chip'>{fmt(row['MAX_CPI'])}</span>\n"
                f"- Unemployment: <span class='metric-chip'>{fmt(row['MAX_UNEMPLOYMENT'])}%</span>\n"
                f"- Mortgage 30Y: <span class='metric-chip'>{fmt(row['MAX_MORTGAGE'])}%</span>")

    if intent == "low":
        row = df.iloc[0]
        return f"🔻 **Lowest {label}**: <span class='metric-chip'>{fmt(row[col])}</span> on **{row['DATE']}**"

    if intent == "low_all":
        row = df.iloc[0]
        return (f"🔻 **All-Time Lows**\n\n"
                f"- CPI: <span class='metric-chip'>{fmt(row['MIN_CPI'])}</span>\n"
                f"- Unemployment: <span class='metric-chip'>{fmt(row['MIN_UNEMPLOYMENT'])}%</span>\n"
                f"- Mortgage 30Y: <span class='metric-chip'>{fmt(row['MIN_MORTGAGE'])}%</span>")

    if intent == "average":
        val = df.iloc[0, 0]
        return f"📊 **Average {label}**: <span class='metric-chip'>{fmt(val)}</span>"

    if intent == "average_all":
        row = df.iloc[0]
        return (f"📊 **Average Indicators**\n\n"
                f"- Avg CPI: <span class='metric-chip'>{fmt(row['AVG_CPI'])}</span>\n"
                f"- Avg Unemployment: <span class='metric-chip'>{fmt(row['AVG_UNEMPLOYMENT'])}%</span>\n"
                f"- Avg Mortgage 30Y: <span class='metric-chip'>{fmt(row['AVG_MORTGAGE'])}%</span>")

    if intent == "meta":
        row = df.iloc[0]
        return (f"🗄️ **Dataset Info**\n\n"
                f"- Total Records: <span class='metric-chip'>{int(row['TOTAL_ROWS']):,}</span>\n"
                f"- From: **{row['EARLIEST_DATE']}**\n"
                f"- To: **{row['LATEST_DATE']}**\n"
                f"- Table: `{TABLE_PATH}`")

    return f"✅ **{explanation}** — {len(df)} rows returned. See table below ↓"


# ─────────────────────────────────────────
# SESSION STATE
# ─────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state["messages"] = []


# ─────────────────────────────────────────
# SIDEBAR — INFO ONLY (no login needed!)
# ─────────────────────────────────────────
with st.sidebar:
    st.markdown("### ❄️ Snowflake Connection")
    st.markdown("""
    <div class='info-box'>
    ✅ <b>Auto-connected!</b><br>
    Running natively inside Snowflake.<br>
    No credentials needed.
    </div>
    """, unsafe_allow_html=True)

    st.divider()
    st.markdown("**📂 Data Source**")
    st.code("ECON_AGENT_DB\n  └─ ANALYTICS\n       └─ ECONOMIC_DASHBOARD_LIVE", language="text")

    st.markdown("**📋 Columns**")
    st.markdown("""
- `DATE` — Date
- `CPI` — Consumer Price Index
- `UNEMPLOYMENT_RATE` — % 
- `MORTGAGE_RATE_30Y` — %
    """)

    st.divider()
    st.markdown("**💬 Example Questions**")
    st.markdown("""
- What is the latest CPI?
- Show unemployment trend 12 months
- Highest mortgage rate ever?
- Average CPI last 6 months
- Show all data for 2023
- Compare all indicators
- How many records in table?
    """)

    st.divider()
    if st.button("🗑️ Clear Chat", use_container_width=True):
        st.session_state["messages"] = []
        st.rerun()


# ─────────────────────────────────────────
# HEADER
# ─────────────────────────────────────────
st.markdown("""
<div class="header-bar">
  <span style="font-size:32px">📊</span>
  <div>
    <div style="font-size:22px;font-weight:700;color:#e8eaf6">Economic Data AI Assistant</div>
    <div style="font-size:13px;color:#5c6bc0">
        Snowflake Native · ECON_AGENT_DB · Rule-Based Engine · No External Connection
    </div>
  </div>
</div>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────
# KPI STRIP — auto loads on startup
# ─────────────────────────────────────────
try:
    latest = run_query(
        f"SELECT DATE, CPI, UNEMPLOYMENT_RATE, MORTGAGE_RATE_30Y FROM {TABLE_PATH} ORDER BY DATE DESC LIMIT 1"
    )
    if not latest.empty:
        row = latest.iloc[0]
        c1, c2, c3, c4 = st.columns(4)
        with c1:
            st.markdown(f'<div class="kpi-card"><div class="kpi-label">📅 Latest Date</div>'
                        f'<div class="kpi-value" style="font-size:18px">{row["DATE"]}</div></div>',
                        unsafe_allow_html=True)
        with c2:
            st.markdown(f'<div class="kpi-card"><div class="kpi-label">📈 CPI</div>'
                        f'<div class="kpi-value">{float(row["CPI"]):.2f}</div></div>',
                        unsafe_allow_html=True)
        with c3:
            st.markdown(f'<div class="kpi-card"><div class="kpi-label">👷 Unemployment</div>'
                        f'<div class="kpi-value">{float(row["UNEMPLOYMENT_RATE"]):.2f}%</div></div>',
                        unsafe_allow_html=True)
        with c4:
            st.markdown(f'<div class="kpi-card"><div class="kpi-label">🏠 Mortgage 30Y</div>'
                        f'<div class="kpi-value">{float(row["MORTGAGE_RATE_30Y"]):.2f}%</div></div>',
                        unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
except Exception as e:
    st.warning(f"⚠️ Could not load KPI strip: {e}")


# ─────────────────────────────────────────
# SUGGESTION PILLS
# ─────────────────────────────────────────
SUGGESTIONS = [
    "What is the latest CPI?",
    "Unemployment trend 12 months",
    "Highest mortgage rate ever?",
    "Economic summary",
    "Average CPI last 6 months",
    "All data for 2023",
    "How many records?",
    "Compare all indicators",
]

if not st.session_state["messages"]:
    st.markdown("**💡 Try asking:**")
    cols = st.columns(4)
    for i, s in enumerate(SUGGESTIONS):
        with cols[i % 4]:
            if st.button(s, key=f"sug_{i}", use_container_width=True):
                st.session_state["pending_question"] = s
                st.rerun()


# ─────────────────────────────────────────
# CHAT HISTORY
# ─────────────────────────────────────────
for msg in st.session_state["messages"]:
    if msg["role"] == "user":
        st.markdown(f'<div class="user-msg">🧑 {msg["content"]}</div>',
                    unsafe_allow_html=True)
    else:
        st.markdown(f'<div class="ai-msg">🤖 {msg["content"]}</div>',
                    unsafe_allow_html=True)
        if msg.get("df") is not None and not msg["df"].empty:
            with st.expander("📋 View Data Table", expanded=False):
                st.dataframe(msg["df"], use_container_width=True, height=220)
        if msg.get("sql"):
            with st.expander("🔍 SQL Query Used", expanded=False):
                st.code(msg["sql"], language="sql")


# ─────────────────────────────────────────
# CHAT INPUT
# ─────────────────────────────────────────
st.markdown("<br>", unsafe_allow_html=True)
col_input, col_send = st.columns([5, 1])
with col_input:
    default_q  = st.session_state.pop("pending_question", "")
    user_input = st.text_input(
        "chat",
        value=default_q,
        placeholder="Ask about CPI, unemployment, mortgage rates…",
        label_visibility="collapsed",
        key="chat_input",
    )
with col_send:
    send = st.button("Send ➤", use_container_width=True)


# ─────────────────────────────────────────
# PROCESS QUESTION
# ─────────────────────────────────────────
if (send or default_q) and user_input.strip():
    question = user_input.strip()
    st.session_state["messages"].append({"role": "user", "content": question})

    with st.spinner("🔍 Querying data…"):
        try:
            intent, params   = classify_intent(question)
            sql, explanation = build_sql(intent, params)
            df               = run_query(sql)
            response_text    = format_response(intent, params, df, explanation)
            st.session_state["messages"].append({
                "role": "assistant",
                "content": response_text,
                "df": df,
                "sql": sql,
            })
        except Exception as e:
            st.session_state["messages"].append({
                "role": "assistant",
                "content": f"❌ **Error:** `{e}`",
            })
    st.rerun()


# ─────────────────────────────────────────
# FOOTER
# ─────────────────────────────────────────
st.markdown("---")
st.markdown(
    f"<div style='text-align:center;color:#37474f;font-size:12px'>"
    f"Economic AI Assistant · Snowflake Native (SiS) · "
    f"Table: <code>{TABLE_PATH}</code>"
    f"</div>",
    unsafe_allow_html=True,